# Creating the fine-tuning dataset for the MSA based on the pre-matched pre-training dataset

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.table import Table
from tqdm import tqdm

pd.options.display.max_columns = None

## Grabbing APOGEE

In [ ]:
data = Table.read("sdss-apogee-dr17.fits")
names = [name for name in data.colnames if len(data[name].shape) <= 1]
data = data[names].to_pandas()
print("APOGEE loaded")

In [ ]:
data.GAIAEDR3_SOURCE_ID.value_counts()

In [ ]:
# removing the duplicated sources and the sources without gaia ids because it will cause issues while finetuning
# potentially could re add them with the full trained model through a wise, 2mass, or other photometric catalogue xmatch
data = data[~data.duplicated(subset="GAIAEDR3_SOURCE_ID", keep=False)]
data.GAIAEDR3_SOURCE_ID.value_counts()

In [ ]:
data

### Adding astroNN ages

In [ ]:
astronn2 = pd.read_csv("nn_latent_age_dr17.csv.gz", compression="gzip")

In [ ]:
astronn2 = astronn2.dropna()
astronn2 = astronn2[
    (astronn2["Age_Error"] / astronn2["Age"] < 0.4)
    & (astronn2["STARFLAG"] == 0)
    & (astronn2["ASPCAPFLAG"] == 0)
]

no STARFLAG and ASPCAPFLAG flag set as well as requiring a latent space age uncertainty less than 40% from Leung2023+

In [ ]:
astronn1 = Table.read("apogee_astroNN-DR17.fits").to_pandas()

In [ ]:
astronn1["APOGEE_ID"] = [val.decode("utf-8") for val in astronn1["APOGEE_ID"].values]

In [ ]:
astronn2 = astronn2.drop_duplicates(subset="APOGEE_ID")
astronn1 = astronn1.drop_duplicates(subset="APOGEE_ID")

In [ ]:
astronn1 = astronn1[~astronn1["APOGEE_ID"].isin(astronn2["APOGEE_ID"])]

In [ ]:
astronn2

In [ ]:
astronn = pd.concat([astronn1, astronn2])

In [ ]:
astronn

In [ ]:
data["APOGEE_ID"] = [val.decode("utf-8") for val in data["APOGEE_ID"].values]

In [ ]:
data = pd.merge(data, astronn, how="left", on="APOGEE_ID")

In [ ]:
data.describe()

In [ ]:
data.GAIAEDR3_SOURCE_ID.dtypes

In [ ]:
data[~np.isnan(data["Age"])].dropna(subset="bp_1")

Having that low data is frustrating

In [ ]:
cols_to_keep = [
    "GAIAEDR3_SOURCE_ID",
    "TEFF_x",
    "TEFF_ERR_x",
    "LOGG_x",
    "LOGG_ERR_x",
    "M_H",
    "M_H_ERR",
    "ALPHA_M",
    "ALPHA_M_ERR",
    "Age",
    "Age_Error",
    "age",
    "age_total_error",
]
renamed_cols = [
    "gaiadr3_source_id",
    "teff",
    "e_teff",
    "logg",
    "e_logg",
    "fe_h",
    "e_fe_h",
    "alpha",
    "e_alpha",
    "age",
    "e_age",
]

In [ ]:
data = data[cols_to_keep]

In [ ]:
data["total_age"] = np.nanmax([data["age"], data["Age"]], axis=0)

In [ ]:
data["total_age_error"] = np.nanmax(
    [data["age_total_error"], data["Age_Error"]], axis=0
)

In [ ]:
cols_to_keep = cols_to_keep[:-4] + ["total_age", "total_age_error"]

In [ ]:
data = data.drop(columns=["Age", "Age_Error", "age", "age_total_error"])

In [ ]:
namedict = {}
for col, newname in zip(cols_to_keep, renamed_cols, strict=False):
    namedict[col] = newname
data = data.rename(columns=namedict)

In [ ]:
data

In [ ]:
gaiadata = []
with h5py.File("220M_pretrain_data.h5", "r") as f:
    progress_bar = tqdm(f.keys(), total=len(f.keys()))
    for key in progress_bar:
        mask = np.isin(f[key][:]["source_id"], data["gaiadr3_source_id"])
        gaiadata.extend([list(tup) for tup in f[key][:][mask]])
    dtypes = f["sslset0_part0"].dtype
    columns = list(dtypes.names)
    f.close()

Using array below here completely turned everything to bytes

In [ ]:
gaiadf = pd.DataFrame(data=gaiadata, columns=columns)

In [ ]:
gaiadf

In [ ]:
cols_to_check = [
    "U_SMSS",
    "E_U_SMSS",
    "V_SMSS",
    "E_V_SMSS",
    "G_SMSS",
    "E_G_SMSS",
    "R_SMSS",
    "E_R_SMSS",
    "I_SMSS",
    "E_I_SMSS",
    "Z_SMSS",
    "E_Z_SMSS",
    "U_SDSS",
    "E_U_SDSS",
    "G_SDSS",
    "E_G_SDSS",
    "R_SDSS",
    "E_R_SDSS",
    "I_SDSS",
    "E_I_SDSS",
    "Z_SDSS",
    "E_Z_SDSS",
]

In [ ]:
cols_to_check = [
    "U_SMSS",
    "E_U_SMSS",
    "V_SMSS",
    "E_V_SMSS",
    "G_SMSS",
    "E_G_SMSS",
    "R_SMSS",
    "E_R_SMSS",
    "I_SMSS",
    "E_I_SMSS",
    "Z_SMSS",
    "E_Z_SMSS",
    "U_SDSS",
    "E_U_SDSS",
    "G_SDSS",
    "E_G_SDSS",
    "R_SDSS",
    "E_R_SDSS",
    "I_SDSS",
    "E_I_SDSS",
    "Z_SDSS",
    "E_Z_SDSS",
    "G_PS1",
    "E_G_PS1",
    "R_PS1",
    "E_R_PS1",
    "I_PS1",
    "E_I_PS1",
    "Z_PS1",
    "E_Z_PS1",
    "Y_PS1",
    "E_Y_PS1",
]

In [ ]:
for col in cols_to_check:
    gaiadf[col] = [np.nan if v in {b"", ""} else float(v) for v in gaiadf[col]]
    gaiadf[col] = gaiadf[col].astype("float64")

In [ ]:
gaiadf

In [ ]:
data = pd.merge(
    data, gaiadf, how="left", left_on="gaiadr3_source_id", right_on="source_id"
)

In [ ]:
data

In [ ]:
data.dtypes

In [ ]:
data = data.drop(columns="source_id")

In [ ]:
data

In [ ]:
data.rename(columns={"gaiadr3_source_id": "source_id"}, inplace=True)

In [ ]:
data

In [ ]:
plt.scatter(data["teff"], data["logg"], c=data["fe_h"], s=0.3)

In [ ]:
plt.scatter(data["teff"], data["logg"], c=data["age"], s=0.3)

In [ ]:
plt.scatter(data["fe_h"], data["alpha"], c=data["age"], s=0.3)

In [ ]:
plt.hexbin(data["fe_h"], data["alpha"], mincnt=1, bins=400)

In [ ]:
del data
del gaiadf
del gaiadata
del astronn1
del astronn2

In [ ]:
towrite = Table.from_pandas(data)
towrite.write("ft_apogee.fits", overwrite=True, format="fits")

## Grabbing GALAH

In [ ]:
data = Table.read("galah_dr4_allstar_240705.fits").to_pandas()

In [ ]:
# same as with APOGEE
data = data[~data.duplicated(subset="gaiadr3_source_id", keep=False)]
data.gaiadr3_source_id.value_counts()

In [ ]:
# checking which columns there are for the next step
data

In [ ]:
# calculating the alpha/fe ratio for the dataset then saving that in a separate column, accounting for the poor measurements

# Define valid flags (only keep 0 and 1)
valid_flags = {0, 1}  # 0 - no problems, 1 - upper limit, all others are pretty drastic

# List of alpha-elements and their corresponding flag columns
alpha_elements = ["mg_fe", "si_fe", "ca_fe", "ti_fe", "o_fe"]
flag_columns = ["flag_mg_fe", "flag_si_fe", "flag_ca_fe", "flag_ti_fe", "flag_o_fe"]
e_alpha_elements = ["e_mg_fe", "e_si_fe", "e_ca_fe", "e_ti_fe", "e_o_fe"]
# Mask out invalid values based on flags
for elem, flag in zip(alpha_elements, flag_columns, strict=False):
    data[elem] = np.where(data[flag].isin(valid_flags), data[elem], np.nan)

# Compute [α/Fe] as the mean of valid elements (ignoring NaNs)
data["alpha"] = data[alpha_elements].mean(axis=1, skipna=True)
data["e_alpha"] = data[e_alpha_elements].mean(axis=1, skipna=True)
data["e_age"] = np.sqrt(
    data["age"]
)  # there are no quoted uncertainties so a sqrt is what I shall use

In [ ]:
# filtering out the numerous columns for lightweight matching
cols_to_keep = [
    "gaiadr3_source_id",
    "teff",
    "e_teff",
    "logg",
    "e_logg",
    "fe_h",
    "e_fe_h",
    "alpha",
    "e_alpha",
    "age",
    "e_age",
]
data = data[cols_to_keep]
data

In [ ]:
gaiadata = []
with h5py.File("220M_pretrain_data.h5", "r") as f:
    progress_bar = tqdm(f.keys(), total=len(f.keys()))
    for key in progress_bar:
        mask = np.isin(f[key][:]["source_id"], data["gaiadr3_source_id"])
        gaiadata.extend([list(tup) for tup in f[key][:][mask]])
    dtypes = f["sslset0_part0"].dtype
    columns = list(dtypes.names)
    f.close()

Using array below here completely turned everything to bytes

In [ ]:
gaiadf = pd.DataFrame(data=gaiadata, columns=columns)

In [ ]:
del gaiadata

In [ ]:
gaiadf

In [ ]:
cols_to_check = [
    "U_SMSS",
    "E_U_SMSS",
    "V_SMSS",
    "E_V_SMSS",
    "G_SMSS",
    "E_G_SMSS",
    "R_SMSS",
    "E_R_SMSS",
    "I_SMSS",
    "E_I_SMSS",
    "Z_SMSS",
    "E_Z_SMSS",
    "U_SDSS",
    "E_U_SDSS",
    "G_SDSS",
    "E_G_SDSS",
    "R_SDSS",
    "E_R_SDSS",
    "I_SDSS",
    "E_I_SDSS",
    "Z_SDSS",
    "E_Z_SDSS",
    "G_PS1",
    "E_G_PS1",
    "R_PS1",
    "E_R_PS1",
    "I_PS1",
    "E_I_PS1",
    "Z_PS1",
    "E_Z_PS1",
    "Y_PS1",
    "E_Y_PS1",
]

In [ ]:
for col in cols_to_check:
    gaiadf[col] = [np.nan if v in {b"", ""} else float(v) for v in gaiadf[col]]
    gaiadf[col] = gaiadf[col].astype("float64")

In [ ]:
gaiadf

In [ ]:
data = pd.merge(
    data, gaiadf, how="left", left_on="gaiadr3_source_id", right_on="source_id"
)

In [ ]:
data

In [ ]:
data = data.drop(columns="source_id")

In [ ]:
data

In [ ]:
data.rename(columns={"gaiadr3_source_id": "source_id"}, inplace=True)

In [ ]:
data

In [ ]:
plt.scatter(data["teff"], data["logg"], c=data["fe_h"], s=0.3)

In [ ]:
plt.scatter(data["teff"], data["logg"], c=data["age"], s=0.3)

In [ ]:
plt.scatter(data["fe_h"], data["alpha"], c=data["age"], s=0.3)

In [ ]:
plt.hexbin(data["fe_h"], data["alpha"], mincnt=1, bins=400)

In [ ]:
del data
del gaiadf

In [ ]:
towrite = Table.from_pandas(data)
towrite.write("ft_galah.fits", overwrite=True, format="fits")

## Combining GALAH and APOGEE

In [ ]:
data = Table.read("ft_apogee.fits").to_pandas()
galah = Table.read("ft_galah.fits").to_pandas()

In [ ]:
data.describe()

In [ ]:
data.dropna(subset="teff", inplace=True)

In [ ]:
galah

In [ ]:
galah.dropna(subset="teff", inplace=True)

In [ ]:
data["spec_source"] = "apogee-latentage"
galah["spec_source"] = "galah"
ft_set = pd.concat([galah, data], ignore_index=True)

In [ ]:
towrite = Table.from_pandas(ft_set)
towrite.write(
    "/arc/projects/k-pop/catalogues/andrae2023/ftset_spec_ga_0602_realmags.fits",
    overwrite=True,
    format="fits",
)

In [ ]:
ft_set.source_id.value_counts()

## Grabbing Li+ 2022 VMPs

In [ ]:
vmps = Table.read("li_et_al_x_gaiaids.fits").to_pandas()

In [ ]:
vmps["source_id"] = vmps["source_id"].str.decode("utf-8").astype(int)

In [ ]:
vmps

In [ ]:
keep_cols = ["source_id", "Teff", "e_Teff", "logg", "e_logg", "FeH", "e_FeH"]
rename_cols = ["source_id", "teff", "e_teff", "logg", "e_logg", "fe_h", "e_fe_h"]
data = vmps[keep_cols]
data = data.rename(
    columns={"Teff": "teff", "e_Teff": "e_teff", "FeH": "fe_h", "e_FeH": "e_fe_h"}
)

In [ ]:
gaiadata = []
with h5py.File("220M_pretrain_data.h5", "r") as f:
    progress_bar = tqdm(f.keys(), total=len(f.keys()))
    for key in progress_bar:
        mask = np.isin(f[key][:]["source_id"], data["source_id"])
        gaiadata.extend([list(tup) for tup in f[key][:][mask]])
    dtypes = f["sslset0_part0"].dtype
    columns = list(dtypes.names)
    f.close()

In [ ]:
gaiadf = pd.DataFrame(data=gaiadata, columns=columns)

In [ ]:
gaiadf

In [ ]:
cols_to_check = [
    "U_SMSS",
    "E_U_SMSS",
    "V_SMSS",
    "E_V_SMSS",
    "G_SMSS",
    "E_G_SMSS",
    "R_SMSS",
    "E_R_SMSS",
    "I_SMSS",
    "E_I_SMSS",
    "Z_SMSS",
    "E_Z_SMSS",
    "U_SDSS",
    "E_U_SDSS",
    "G_SDSS",
    "E_G_SDSS",
    "R_SDSS",
    "E_R_SDSS",
    "I_SDSS",
    "E_I_SDSS",
    "Z_SDSS",
    "E_Z_SDSS",
]

In [ ]:
cols_to_check = [
    "U_SMSS",
    "E_U_SMSS",
    "V_SMSS",
    "E_V_SMSS",
    "G_SMSS",
    "E_G_SMSS",
    "R_SMSS",
    "E_R_SMSS",
    "I_SMSS",
    "E_I_SMSS",
    "Z_SMSS",
    "E_Z_SMSS",
    "U_SDSS",
    "E_U_SDSS",
    "G_SDSS",
    "E_G_SDSS",
    "R_SDSS",
    "E_R_SDSS",
    "I_SDSS",
    "E_I_SDSS",
    "Z_SDSS",
    "E_Z_SDSS",
    "G_PS1",
    "E_G_PS1",
    "R_PS1",
    "E_R_PS1",
    "I_PS1",
    "E_I_PS1",
    "Z_PS1",
    "E_Z_PS1",
    "Y_PS1",
    "E_Y_PS1",
]

In [ ]:
for col in cols_to_check:
    gaiadf[col] = [np.nan if v in {b"", ""} else float(v) for v in gaiadf[col]]
    gaiadf[col] = gaiadf[col].astype("float64")

In [ ]:
gaiadf

In [ ]:
data = pd.merge(data, gaiadf, how="left", left_on="source_id", right_on="source_id")

In [ ]:
data.dropna(subset="G", inplace=True)
data

In [ ]:
data.dtypes

In [ ]:
plt.scatter(data["teff"], data["logg"], c=data["fe_h"], s=0.3)

In [ ]:
ft_set

In [ ]:
data["spec_source"] = "li_et_al_vmps"
ft_set2 = pd.concat([ft_set, data], ignore_index=True)

In [ ]:
ft_set2

In [ ]:
towrite = Table.from_pandas(ft_set2)
towrite.write(
    "/arc/projects/k-pop/catalogues/andrae2023/ftset_spec_ga_0602_realmags.fits",
    overwrite=True,
    format="fits",
)

In [ ]:
ft_set2.source_id.value_counts()

In [ ]:
plt.scatter(ft_set2["teff"], ft_set2["logg"], c=ft_set2["fe_h"], s=0.3)

In [ ]:
plt.scatter(ft_set2["teff"], ft_set2["logg"], c=ft_set2["age"], s=0.3)

In [ ]:
plt.scatter(ft_set2["fe_h"], ft_set2["alpha"], c=ft_set2["age"], s=0.3)

In [ ]:
plt.hexbin(ft_set2["fe_h"], ft_set2["alpha"], mincnt=1, bins=400)

## Ensuring there are no NaNs

In [ ]:
ft_set = Table.read(
    "/arc/projects/k-pop/catalogues/andrae2023/ftset_spec_ga_0602_realmags.fits"
).to_pandas()

In [ ]:
ft_set = ft_set.dropna(subset=["G"])

In [ ]:
ft_set

In [ ]:
towrite = Table.from_pandas(ft_set)
towrite.write(
    "/arc/projects/k-pop/catalogues/andrae2023/ftset_spec_ga_0602_realmags.fits",
    overwrite=True,
    format="fits",
)